In [11]:
import uuid
from datetime import datetime, timezone
import json
from openlineage.client.run import RunEvent, RunState, Run, Job, InputDataset
from openlineage.client.facet import SchemaDatasetFacet, SchemaField

# 1. Define your dataset (using InputDataset for maximum stability)
my_dataset = InputDataset(
    namespace="postgresql://dwh-server:5432/production",
    name="analytics.users_dim",
    facets={
        "schema": SchemaDatasetFacet(
            fields=[
                SchemaField(name="user_id", type="INT", description="Primary Key"),
                SchemaField(name="email", type="VARCHAR", description="User Email")
            ]
        )
    }
)

# 2. Build the mandatory metadata wrappers (Job and Run)
# Every OpenLineage event requires a Job and a Run context
job = Job(
    namespace="dataset_inventory",
    name="register_users_dim_metadata"
)

run = Run(
    runId=str(uuid.uuid4()),
    facets={}
)


In [13]:

PRODUCER_URI = "https://github.com"

# 3. Create the top-level OpenLineage Document (RunEvent)
# We log it as a COMPLETE event containing our dataset in the inputs list
openlineage_document = RunEvent(
    eventType=RunState.COMPLETE,
    eventTime=datetime.now(timezone.utc).isoformat(),
    run=run,
    job=job,
    inputs=[my_dataset],
    outputs=[],
    producer=PRODUCER_URI
)


In [15]:
# 6. Печатаем результат в формате JSON (документ готов к отправке в Marquez / Airflow)
import json
from openlineage.client.serde import Serde

# Сгенерировать JSON-строку (String)
json_string = Serde.to_json(openlineage_document)


# 4. Serialize to JSON format
print(json.dumps(json.loads(json_string), indent=2, ensure_ascii=False))

{
  "eventTime": "2026-06-25T13:03:34.977746+00:00",
  "eventType": "COMPLETE",
  "inputs": [
    {
      "facets": {
        "schema": {
          "_producer": "https://github.com/OpenLineage/OpenLineage/tree/1.16.0/client/python",
          "_schemaURL": "https://raw.githubusercontent.com/OpenLineage/OpenLineage/main/spec/OpenLineage.json#/definitions/SchemaDatasetFacet",
          "fields": [
            {
              "description": "Primary Key",
              "name": "user_id",
              "type": "INT"
            },
            {
              "description": "User Email",
              "name": "email",
              "type": "VARCHAR"
            }
          ]
        }
      },
      "inputFacets": {},
      "name": "analytics.users_dim",
      "namespace": "postgresql://dwh-server:5432/production"
    }
  ],
  "job": {
    "facets": {},
    "name": "register_users_dim_metadata",
    "namespace": "dataset_inventory"
  },
  "outputs": [],
  "producer": "https://github.com",
 